# Demo: flujo de delegacion del Orquestador Multi-Agente

Pre-entrega 6. Este notebook corre el grafo real (Supervisor + Investigador + Analista) contra la API de Groq y muestra, paso a paso, cada decision del Supervisor y cada aporte de los especialistas - la evidencia concreta del flujo de delegacion jerarquico.

Requiere `GROQ_API_KEY` en el `.env` de la raiz del proyecto.

In [1]:
import sys
from pathlib import Path

# Hace importable la raiz del proyecto (fase3_ejercicio_chromadb, fase3_rag_local, etc.)
sys.path.insert(0, str(Path.cwd().parent))

from langchain_core.messages import HumanMessage

from fase6_orquestador_multiagente.graph import compile_graph

app = compile_graph()
print("Grafo compilado: nodos =", list(app.get_graph().nodes.keys()))

Grafo compilado: nodos = ['__start__', 'supervisor', 'investigador', 'analista', '__end__']


## Pregunta que obliga a delegar en ambos especialistas

Requiere investigar un dato factual (los tiempos de SLA) Y despues procesarlo numericamente (convertir unidades y promediar) - el caso que la consigna pide probar explicitamente: Supervisor -> Investigador -> Supervisor -> Analista -> Supervisor -> FINISH.

In [2]:
pregunta = (
    "Investiga en la base de conocimiento los tiempos de respuesta "
    "comprometidos por severidad de soporte, y calculame el promedio en "
    "horas."
)
print("Usuario:", pregunta)

Usuario: Investiga en la base de conocimiento los tiempos de respuesta comprometidos por severidad de soporte, y calculame el promedio en horas.


## Corrida paso a paso (`astream`)

Usamos `astream` en vez de `ainvoke` para poder imprimir la decision del Supervisor y el aporte de cada especialista a medida que el grafo avanza, en vez de solo ver el resultado final.

In [3]:
estado_inicial = {
    "messages": [HumanMessage(content=pregunta)],
    "next_agent": None,
    "task_completed": False,
    "contribuciones": [],
}

paso_num = 0
estado_final = None
async for paso in app.astream(estado_inicial, config={"recursion_limit": 15}):
    for nodo, actualizacion in paso.items():
        paso_num += 1
        print(f"\n{'='*70}\nPaso {paso_num} - Nodo: {nodo}\n{'='*70}")
        if actualizacion.get("next_agent"):
            print(f"  -> Supervisor decide rutear a: {actualizacion['next_agent']}")
        for m in actualizacion.get("messages", []):
            autor = getattr(m, "name", None) or nodo
            print(f"  [{autor}] {m.content[:400]}")
        estado_final = actualizacion

12:24:11 | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


12:24:11 | Supervisor decide: investigador (Falta obtener la información factual de los tiempos de respuesta por severidad de soporte antes de poder calcular el promedio.)



Paso 1 - Nodo: supervisor
  -> Supervisor decide rutear a: investigador


12:24:12 | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


12:24:14 | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


...

12:24:27 | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"


12:24:27 | Retrying request to /openai/v1/chat/completions in 10.000000 seconds


12:24:38 | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


12:24:38 | Supervisor decide: FINISH (Ya se investigaron los tiempos de respuesta por severidad y el analista calculó el promedio en horas; la tarea está completa.)



Paso 5 - Nodo: supervisor
  -> Supervisor decide rutear a: FINISH
  [supervisor] Según la política de soporte, los tiempos de respuesta comprometidos son: Severidad 1 – menos de 30 min (0.5 h), Severidad 2 – menos de 4 h hábiles (4 h), Severidad 3 – menos de 2 días hábiles (≈16 h). El promedio de estos tiempos es aproximadamente **6.83 horas**.


## Estado final: respuesta sintetizada + registro de contribuciones

El Supervisor combina lo que aporto cada especialista en una unica respuesta final (la "fase de sintesis" que pide la consigna). El registro de `contribuciones` muestra, en orden, que especialista aporto que - asi no se pierde el rastro de quien dijo que en la conversacion.

In [4]:
# El estado completo (no solo el delta del ultimo nodo) se obtiene volviendo a invocar
# sobre el mismo grafo sin checkpointer no es posible releer el historial completo;
# para este notebook alcanza con re-invocar con ainvoke para ver el resultado consolidado.
resultado = await app.ainvoke(estado_inicial, config={"recursion_limit": 15})

print("=== Respuesta final (sintesis del Supervisor) ===")
print(resultado["messages"][-1].content)

print("\n=== Contribuciones registradas ===")
for c in resultado["contribuciones"]:
    print(f"  [{c['agente']}] {c['resumen'][:200]}...")

print("\ntask_completed:", resultado["task_completed"])

12:24:39 | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"


12:24:39 | Retrying request to /openai/v1/chat/completions in 5.000000 seconds


12:24:45 | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


12:24:45 | Supervisor decide: investigador (Necesitamos obtener los tiempos de respuesta comprometidos por severidad de soporte desde la base de conocimiento antes de poder calcular el promedio.)


12:24:46 | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"


...

12:25:21 | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"


12:25:21 | Retrying request to /openai/v1/chat/completions in 9.000000 seconds


12:25:32 | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


12:25:32 | Supervisor decide: FINISH (Ya se investigaron los tiempos de respuesta y el analista calculó el promedio en horas, por lo que la tarea está completa.)


=== Respuesta final (sintesis del Supervisor) ===
Los tiempos de respuesta comprometidos son: Severidad 1 – menos de 30 min (0.5 h), Severidad 2 – menos de 4 h hábiles (4 h), Severidad 3 – menos de 2 días hábiles (≈16 h). El promedio de estos tiempos es aproximadamente **6.83 horas**.

=== Contribuciones registradas ===
  [investigador] **Tiempos de respuesta comprometidos (SLA) según severidad de soporte**

| Severidad | Tiempo de respuesta comprometido |
|-----------|----------------------------------|
| **Severidad 1** (sistema ca...
  [analista] He convertido los tiempos de respuesta a una unidad común (horas):

- **Severidad 1:** 30 min = 0.5 h  
- **Severidad 2:** 4 h hábiles = 4 h  
- **Severidad 3:** 2 días hábiles ≈ 2 × 8 h = 16 h  

Con...

task_completed: True


## Conclusion

El flujo de delegacion observado en esta corrida:

1. **Supervisor -> investigador**: falta informacion factual (los tiempos de SLA no estan en la pregunta).
2. **Investigador** busca en la Vector DB (Fase 3) y encuentra la politica de soporte.
3. **Supervisor -> analista**: ya hay datos, falta procesarlos.
4. **Analista** convierte unidades y calcula el promedio con la herramienta `calcular_estadisticas`.
5. **Supervisor -> FINISH**: sintetiza la respuesta final combinando ambos aportes.

Ver tambien `traces/ejemplo_flujo_delegacion.json` (la misma corrida, en formato JSON) y `README.md` (arquitectura completa + diagrama Mermaid del grafo).